In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection  import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("synthetic_road_accidents.csv")

In [ ]:
df

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,rural,2,0.29,70.0,night,rainy,False,True,evening,False,False,1.0,0.64
1,highway,1,0.34,25.0,dim,clear,False,False,morning,False,False,3.0,0.27
2,rural,2,0.76,70.0,night,foggy,True,False,evening,True,True,1.0,0.76
3,rural,3,0.37,70.0,night,foggy,True,False,morning,False,True,0.0,0.60
4,highway,3,0.39,45.0,dim,rainy,False,True,morning,False,False,0.0,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15772,highway,1,0.18,60.0,night,foggy,False,False,evening,False,False,1.0,0.61
15773,highway,3,0.51,60.0,night,foggy,True,False,morning,True,True,2.0,0.66
15774,highway,2,0.49,35.0,night,rainy,True,False,evening,True,True,2.0,0.40
15775,urban,4,0.40,25.0,night,rainy,False,True,morning,True,True,0.0,0.43


In [ ]:
#X = ['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'time_of_day', 'holiday', 'school_season', 'num_reported_accidents']
#Y = ['accident_risk']

In [ ]:
#OneHotEncoding
categorical_features = ['road_type', 'lighting', 'weather', 'time_of_day']
boolean_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']
numerical_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
df_encoded = pd.get_dummies(df, columns=categorical_features)

In [ ]:
df_encoded

,num_lanes,curvature,speed_limit,road_signs_present,public_road,holiday,school_season,num_reported_accidents,accident_risk,road_type_highway,...,road_type_urban,lighting_daylight,lighting_dim,lighting_night,weather_clear,weather_foggy,weather_rainy,time_of_day_afternoon,time_of_day_evening,time_of_day_morning
0,2,0.29,70.0,False,True,False,False,1.0,0.64,False,...,False,False,False,True,False,False,True,False,True,False
1,1,0.34,25.0,False,False,False,False,3.0,0.27,True,...,False,False,True,False,True,False,False,False,False,True
2,2,0.76,70.0,True,False,True,True,1.0,0.76,False,...,False,False,False,True,False,True,False,False,True,False
3,3,0.37,70.0,True,False,False,True,0.0,0.60,False,...,False,False,False,True,False,True,False,False,False,True
4,3,0.39,45.0,False,True,False,False,0.0,0.17,True,...,False,False,True,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15772,1,0.18,60.0,False,False,False,False,1.0,0.61,True,...,False,False,False,True,False,True,False,False,True,False
15773,3,0.51,60.0,True,False,True,True,2.0,0.66,True,...,False,False,False,True,False,True,False,False,False,True
15774,2,0.49,35.0,True,False,True,True,2.0,0.40,True,...,False,False,False,True,False,False,True,False,True,False
15775,4,0.40,25.0,False,True,True,True,0.0,0.43,False,...,True,False,False,True,False,False,True,False,False,True


In [ ]:
df_encoded = pd.get_dummies(df_encoded, columns=boolean_features)

In [ ]:
df_numerical = df[numerical_features]

In [ ]:
DF = pd.concat([df_encoded, df_numerical], axis = 1)

In [ ]:
X = DF.drop('accident_risk', axis = 1)
Y = DF['accident_risk']

In [ ]:
#Train_Test_Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

In [ ]:
# Convert to binary
Y_train_binary = (Y_train >= 0.50).astype(int)
Y_test_binary = (Y_test >= 0.50).astype(int)

In [ ]:
Y_train_binary

,accident_risk
4330,1
14670,1
15002,0
8940,0
7938,0
...,...
3773,0
15608,0
5520,0
11487,0


In [ ]:
#Model Training
model = LogisticRegression()
model.fit(X_train, Y_train_binary)

In [ ]:
# Predict
y_pred = model.predict(X_test)

print(y_pred)


[0 0 1 ... 0 0 0]


In [ ]:
#Metric Values
accuracy = accuracy_score(Y_test_binary, y_pred)
cr = classification_report(Y_test_binary, y_pred)
print("Accuracy:", accuracy)
print(cr)

Accuracy: 0.90895
              precision    recall  f1-score   support

           0       0.93      0.95      0.94     14612
           1       0.85      0.81      0.83      5388

    accuracy                           0.91     20000
   macro avg       0.89      0.88      0.88     20000
weighted avg       0.91      0.91      0.91     20000



In [ ]:

# Predictions
Y_train_pred = model.predict(X_train)

# Accuracy
train_acc = accuracy_score(Y_train_binary, Y_train_pred)

print(f"Train Accuracy: {train_acc:.4f}")

# Optional: full report
print("\nClassification Report (Test Data):")

In [ ]:
#Hyperparamenters
from sklearn.model_selection import GridSearchCV, StratifiedKFold
# Basic parameter grid for binary classification
param_grid = {
    'penalty': ['l1', 'l2'],  # Regularization type
    'C': [0.001, 0.01, 0.1, 1, 10, 100],   # Inverse regularization strength
    'solver': ['liblinear', 'saga'],        # Solvers that support L1/L2/elasticnet
    'class_weight': [None, 'balanced'],     # Handle class imbalance
    'max_iter': [1000, 2000]                # Ensure convergence
}

In [ ]:
# Initialize model
model = LogisticRegression(random_state=42)

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring metrics
scoring = {
    'accuracy': 'accuracy'
}

# Perform GridSearch
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',  # Primary metric to optimize
    refit=True,    # Refit best model on entire training set
    n_jobs=-1,     # Use all available cores
    verbose=1      # Show progress
)

# Fit the grid search
print("Starting GridSearchCV...")
grid_search.fit(X_train, Y_train_binary)

Starting GridSearchCV...
Fitting 5 folds for each of 96 candidates, totalling 480 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'max_iter': [1000, 2000], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'saga']},
             scoring='accuracy', verbose=1)

In [ ]:
print("=== GRIDSEARCH RESULTS ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
print(f"Best estimator: {grid_search.best_estimator_}")

# Get all results
results_df = pd.DataFrame(grid_search.cv_results_)
print(f"\nTotal models tested: {len(results_df)}")

In [ ]:
#XG_Boost {CV}
from xgboost import XGBClassifier

In [ ]:
np.array(Y)
X_train_np = np.array(X_train)
y_train_np = np.array(Y_train_binary).flatten()
print(X_train_np)
y_train_np

[[1 0.97 25 ... 0.97 25 1]
 [3 0.94 45 ... 0.94 45 2]
 [4 0.76 70 ... 0.76 70 0]
 ...
 [1 0.52 45 ... 0.52 45 1]
 [1 0.04 25 ... 0.04 25 0]
 [2 0.83 70 ... 0.83 70 1]]


array([0, 0, 0, ..., 1, 0, 0])

In [ ]:
print(f"Data shapes - X: {X_train_np.shape}, y: {y_train_np.shape}")
print(f"Data types - X: {X_train_np.dtype}, y: {y_train_np.dtype}")


Data shapes - X: (80000, 28), y: (80000,)
Data types - X: object, y: int64


In [ ]:
# Define model
model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# Parameter grid
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

# Stratified K-Fold for classification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
grid_search.fit(X_train_np, y_train_np)

# Results
print("Best parameters:", grid_search.best_params_)
print("Best CV score: {:.4f}".format(grid_search.best_score_))

# Get best model
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 16 candidates, totalling 80 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:39:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
Best CV score: 0.7291


In [ ]:
Y_train_binary

,accident_risk
1370,0
21729,0
17514,0
68813,0
77187,1
...,...
69965,1
54288,0
11670,1
9605,0


In [ ]:
#LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer

# Initialize the base classifier
lgbm = LGBMClassifier(random_state=42, verbose=-1)  # verbose=-1 suppresses output

# Define the parameter grid
param_grid = {
    'num_leaves': [15, 31],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

# Create scorer
scorer = make_scorer(accuracy_score)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,  # use all available cores
    verbose=1  # prints progress
)

# Fit the grid search
grid_search.fit(X_train_np, y_train_np)

# Best parameters and score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Evaluate on test set
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 50, 'num_leaves': 15, 'subsample': 0.8}
Best cross-validation score: 0.7291


In [ ]:
#Feature Engineering
df['speed_ratio'] = df['vehicle_speed'] / df['speed_limit']
df['overspeed_flag'] = (df['vehicle_speed'] > df['speed_limit']).astype(int)

df['bad_weather'] = df['weather_condition'].isin(['Rain', 'Snow', 'Fog']).astype(int)



In [ ]:
#DeepLearning{Neural_Network}

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # single neuron for binary output
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])